# 1장 2강: 데이터 추출과 저장 — 실습문제


## 실습 목표

- 분석 목적에 맞는 필드를 선정하고, 한 행이 무엇을 의미하는지 정할 수 있다.
- BeautifulSoup의 `select()`, `select_one()`, `get_text()`로 HTML에서 값을 추출할 수 있다.
- `json.loads()`와 Key 접근으로 중첩 JSON에서 필요한 값을 추출할 수 있다.
- 추출한 값을 동일한 Key를 가진 Dictionary로 만들고 List에 모을 수 있다.
- DataFrame의 행 수·컬럼·자료형을 확인하고 CSV로 저장할 수 있다.
- 저장한 CSV를 다시 읽어 행 수·컬럼·데이터가 유지되었는지 확인할 수 있다.

## 실습 환경 / 데이터

- 환경: Python 3, Jupyter Notebook 또는 Google Colab
- 사용 라이브러리: `json`, `pathlib`(Python 기본 모듈), `beautifulsoup4`, `pandas`
- 기준 교안: **1장 2강: 데이터 추출과 저장**
- 사례 사이트: [Quotes to Scrape](https://quotes.toscrape.com/)
- JSON 구조 참고 주소: [명언 JSON](https://quotes.toscrape.com/api/quotes?page=1)
- 데이터: 교안의 **가상 명언 두 개를 표현한 HTML·JSON**과 교안 연습문제의 명언 A·B·C를 같은 구조로 표현한 과제용 HTML입니다.
- 모든 입력은 아래 셀에 포함되어 있습니다. 기존 CSV 파일이나 실시간 사이트 접속 없이 실행할 수 있습니다.

> 이번 파일은 교안에 있는 사례와 기능을 우선합니다. 이전 사이트 배정표의 책 예시 대신 **교안의 명언 예시**를 사용합니다. 아래 URL은 교안과 같이 출처 기록 방법을 설명하기 위한 예시 값입니다. 가상 명언이 실제 사이트에 게시되어 있거나 이 노트북이 해당 주소에 요청했다는 뜻은 아닙니다.

| 문제 | 사용할 입력 | 수행할 일 | 결과물 |
|---|---|---|---|
| 필수 1 | `html_text` | 필드 설계 → HTML 추출 → List와 Dictionary 구성 | `html_records`, 필드 설계표 |
| 필수 2 | `json_text` | JSON 추출 → HTML 결과와 비교 → DataFrame → CSV → 재확인 | `quotes_example.csv` |
| 과제 1 | `assignment_html` | 다른 명언 세 개에 HTML 추출·저장 과정을 적용 | `quotes_assignment.csv` |

필수 1·2와 과제 1은 기초~중급 수준입니다. 과제는 필수 문제의 방법을 다른 값에 적용하며 새로운 기능을 요구하지 않습니다. 이번 실습에서는 페이지네이션, XPath, 동적 크롤링, DB 저장, 결측·중복 처리 코드를 다루지 않습니다.


## 시작 전에 읽을 기초 개념

1장 1강에서 HTML·JSON의 **구조와 값의 위치**를 살펴보았다면, 이번에는 그 위치의 값을 실제로 꺼내서 표와 파일로 만듭니다. 목표는 **명언 하나에 해당하는 정보를 한 행에 모으고, 저장한 뒤 다시 읽어 확인하는 것**입니다.

### 용어를 이번 예시에 연결하기

| 용어 | 쉽게 풀어쓴 뜻 | 이번 실습에서의 쓰임 |
|---|---|---|
| 필드(Field) | 대상의 특징 하나를 담는 항목 | 본문 text, 작성자 author |
| 레코드(Record) | 대상 하나의 정보를 묶은 것 | 한 명언의 본문·작성자·태그·출처를 담은 dict |
| 스키마(Schema) | 어떤 항목을 어떤 이름·형태로 담을지 정한 구조 | 네 컬럼의 이름·순서와 저장 값의 형태 |
| 필수 필드 | 분석 목적에 꼭 필요한 항목 | 본문을 분석하려면 text가 필요 |
| 메타데이터 | 데이터의 출처 등을 설명하는 부가 정보 | source_url에 기록한 출처 주소 |
| 파싱(Parsing) | 텍스트의 구조를 프로그램이 다룰 수 있도록 해석 | HTML은 BeautifulSoup, JSON 문자열은 json.loads() |
| 추출(Extraction) | 해석한 구조에서 필요한 값을 꺼내는 일 | 명언 영역에서 본문 글자만 가져오기 |
| CSS 선택자 | HTML에서 원하는 요소를 찾는 조건 | div.quote는 quote 클래스를 가진 div 요소 |
| HTML 요소 | 태그와 그 안의 내용 등으로 이루어진 구성 단위 | `<small class="author">작성자 A</small>` |
| 문자열(str) | 글자들이 들어 있는 값 | 작성자 A, life, thinking |
| 딕셔너리(dict) | 이름인 Key와 실제 값인 Value를 짝지은 묶음 | `{"author": "작성자 A"}` |
| 리스트(list) | 여러 항목을 순서대로 담는 목록 | 명언별 dict를 모은 html_records |
| 반복문 | 목록의 항목마다 같은 처리를 실행하는 코드 | 명언마다 본문·작성자·태그 추출 |
| DataFrame | Python에서 다루는 행·열 형태의 표 | 명언은 행, 네 필드는 열 |
| CSV | 표의 값을 텍스트로 기록한 파일 형식 | quotes_example.csv |
| 인덱스(Index) | DataFrame에서 행을 구분하는 표지 | 이번 예시의 왼쪽 0, 1 |
| 인코딩(Encoding) | 글자를 파일에 저장하고 읽는 표현 방식 | 한글 저장·읽기에 사용하는 utf-8-sig |

**파싱과 추출을 나누어 이해하세요.** BeautifulSoup으로 구조를 해석했다고 본문만 자동으로 선택되는 것은 아닙니다. 이어서 select_one()으로 요소를 찾고 get_text()로 글자를 얻습니다. JSON도 json.loads()로 해석한 다음 Key로 필요한 값에 접근합니다.

### 한 행과 한 칸을 먼저 정하는 이유

이번 표의 기준은 **명언 하나 = 한 행**입니다. 같은 작성자의 명언도 본문이 다른 별개의 명언이면 각각 한 행으로 남깁니다. 한 명언에 태그가 여러 개 있어도 그 태그들을 한 문자열로 연결해 tags 한 칸에 담습니다.

최종 필드 이름을 먼저 정하면 HTML에서 꺼낸 값과 JSON에서 꺼낸 값을 같은 형태로 맞출 수 있습니다. 원본의 작성자 정보가 중첩되어 있어도 최종 author에는 이름 문자열을 넣습니다. 원본에 없는 source_url은 출처를 기록하기 위해 프로그램에서 추가합니다.

### 전체 실습 흐름

| 단계 | 하는 일 | 작업 후 남는 것 |
|---|---|---|
| 준비 | 제공된 HTML·JSON 글자를 변수에 보관 | 원시 문자열 |
| 필드 설계 | 한 행의 기준과 네 필드의 의미를 정함 | 필드 설계표 |
| 구조 해석·값 추출 | HTML 요소 또는 JSON Key를 따라 값을 꺼냄 | 본문·작성자·태그 등의 값 |
| 공통 구조 구성 | 명언 하나는 dict, 여러 명언은 list로 묶음 | html_records 또는 json_records |
| 표로 변환 | 레코드 목록을 DataFrame으로 만듦 | 행·열로 정리된 df |
| 파일 저장 | to_csv()로 표의 값을 기록 | CSV 파일 |
| 다시 읽어 확인 | read_csv()로 읽은 뒤 저장 전과 비교 | 행 수·컬럼·내용의 일치 여부 |

필수 1의 HTML 결과와 필수 2의 JSON 결과는 같은 명언의 표현 방식을 비교하기 위한 것입니다. 필수 2에서 파일로 저장하는 입력은 **json_records 하나**입니다.

### 코드를 읽을 때 구분할 기호

| 표현 | 읽는 방법 | 주의할 점 |
|---|---|---|
| `이름 = 값` | 오른쪽 값을 왼쪽 이름에 보관 | `==`와 역할이 다름 |
| `왼쪽 == 오른쪽` | 두 값이 같은지 비교 | 결과는 True 또는 False |
| `"text"`와 `text` | 글자 text / text라는 변수에 담긴 값 | 따옴표 유무를 구분 |
| `{"text": text}` | text라는 Key에 변수 text의 값을 넣음 | 왼쪽은 항목 이름, 오른쪽은 실제 내용 |
| `[]`와 `{}` | 빈 list / 빈 dict | 여러 레코드를 모을 때는 list 사용 |
| `목록.append(항목)` | 목록 끝에 항목 하나 추가 | 이번 항목은 명언 하나의 dict |
| `for 항목 in 목록:` | 목록에서 하나씩 꺼내 반복 | 들여쓴 부분이 반복할 작업 |
| `print(값)` | 화면에 값을 표시 | 파일로 저장하는 동작은 아님 |
| `df.shape`와 `df.head()` | 크기 정보 확인 / 앞부분 조회 실행 | 정답 코드의 괄호 유무를 그대로 구분 |

같은 변수 이름 quote도 HTML 반복에서는 **현재 명언의 HTML 요소**, JSON 반복에서는 **현재 명언의 dict**를 가리킵니다. 이름 자체가 자료형을 결정하지 않으므로 어디에서 가져온 값인지 함께 읽어야 합니다.


## 실습 준비

1. 아래 제공 셀을 실행하여 라이브러리와 HTML·JSON 문자열을 준비하세요.
2. `html_text`와 `json_text`가 같은 명언 두 개를 표현하는지 읽어 보세요.
3. HTML의 `div.quote` 안에 본문·작성자·태그가 있고, JSON의 `quotes` 목록 안에는 `author` 객체가 중첩되어 있음을 확인하세요.
4. 아래 순서대로 문제를 해결하세요. 필수 2의 비교 코드는 필수 1의 `html_records`를 사용합니다. 과제는 준비 셀의 데이터만으로 해결할 수 있습니다.

**준비 셀은 수정하지 않습니다.** `bs4` 또는 `pandas`를 찾을 수 없다는 오류가 나오면 해당 라이브러리가 설치된 실습 환경을 사용하세요. CSV 파일은 코드를 실행하는 현재 작업 폴더에 저장됩니다. 같은 코드를 다시 실행하면 같은 이름의 CSV가 갱신됩니다.

HTML을 해석할 때 쓰는 선택자는 다음과 같습니다. 교안에서 사용한 선택자와 동일합니다.

| 대상 | 선택자 |
|---|---|
| 명언 한 개의 영역 | `div.quote` |
| 명언 본문 | `span.text` |
| 작성자 이름 | `small.author` |
| 관련 태그 각각 | `a.tag` |


In [1]:
import json
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd

# 교안의 가상 예시입니다. 실제 요청을 보내는 코드가 아닙니다.
HTML_SOURCE_URL = "https://quotes.toscrape.com/"
JSON_SOURCE_URL = "https://quotes.toscrape.com/api/quotes?page=1"

html_text = """
<div class="quote">
    <span class="text">첫 번째 예시 명언입니다.</span>
    <small class="author">작성자 A</small>
    <div class="tags">
        <a class="tag">life</a>
        <a class="tag">thinking</a>
    </div>
</div>
<div class="quote">
    <span class="text">두 번째 예시 명언입니다.</span>
    <small class="author">작성자 B</small>
    <div class="tags">
        <a class="tag">learning</a>
    </div>
</div>
"""

json_text = """
{
  "quotes": [
    {
      "text": "첫 번째 예시 명언입니다.",
      "author": {"name": "작성자 A"},
      "tags": ["life", "thinking"]
    },
    {
      "text": "두 번째 예시 명언입니다.",
      "author": {"name": "작성자 B"},
      "tags": ["learning"]
    }
  ]
}
"""

# 교안 연습문제의 명언 A·B·C를 같은 HTML 구조로 표현했습니다.
assignment_html = """
<div class="quote">
    <span class="text">명언 A</span>
    <small class="author">작성자 1</small>
    <div class="tags">
        <a class="tag">life</a>
        <a class="tag">learning</a>
    </div>
</div>
<div class="quote">
    <span class="text">명언 B</span>
    <small class="author">작성자 2</small>
    <div class="tags">
        <a class="tag">thinking</a>
    </div>
</div>
<div class="quote">
    <span class="text">명언 C</span>
    <small class="author">작성자 1</small>
    <div class="tags">
        <a class="tag">success</a>
        <a class="tag">life</a>
    </div>
</div>
"""

print("실습 준비 완료: 필수 문제용 HTML·JSON, 과제용 HTML")
print("HTML 출처 기록 예시:", HTML_SOURCE_URL)
print("JSON 출처 기록 예시:", JSON_SOURCE_URL)


실습 준비 완료: 필수 문제용 HTML·JSON, 과제용 HTML
HTML 출처 기록 예시: https://quotes.toscrape.com/
JSON 출처 기록 예시: https://quotes.toscrape.com/api/quotes?page=1


### 준비 코드 상세 해설 — 입력과 도구 구분하기

| 코드·변수 | 역할 | 아직 하지 않은 일 |
|---|---|---|
| `import json` | JSON 문자열을 해석할 도구를 준비 | JSON 값 추출 |
| `from pathlib import Path` | 파일 경로를 표현할 도구를 준비 | CSV 파일 생성 |
| `from bs4 import BeautifulSoup` | HTML을 해석할 도구를 준비 | 명언 본문 선택 |
| `import pandas as pd` | pandas를 pd라는 짧은 이름으로 사용 | DataFrame 생성 |
| `HTML_SOURCE_URL`, `JSON_SOURCE_URL` | 출처 기록용 예시 주소를 문자열로 보관 | 웹사이트 접속 |
| `html_text` | 필수 문제의 두 명언을 HTML 문자열로 보관 | HTML 파싱·추출 |
| `json_text` | 같은 두 명언을 JSON 문자열로 보관 | dict·list로 변환 |
| `assignment_html` | 과제의 세 명언을 HTML 문자열로 보관 | 과제 레코드 생성 |
| 마지막 print()들 | 준비 완료 문구와 예시 주소를 화면에 표시 | 파일 저장 |

큰따옴표 세 개로 둘러싼 부분은 **여러 줄을 담은 하나의 문자열**입니다. 줄마다 별도 명언이 되는 것이 아닙니다. HTML에서는 div.quote 영역 하나, JSON에서는 quotes 목록 안의 객체 하나가 한 명언을 나타냅니다.

노트북은 준비 → 필수 1 → 필수 2 → 과제 순서로 실행하세요. 필수 2에서 html_records를 비교에 사용하므로 필수 1을 먼저 실행해야 합니다. 과제의 입력은 assignment_html이며, 필수 문제의 html_text와 구분합니다.

**주소의 의미도 구분하세요.** 두 URL은 이 가상 예시에서 출처를 기록하는 연습에 사용합니다. 이 노트북에는 실제 요청 코드가 없으므로 URL을 출력했다고 웹에서 새 데이터를 받은 것은 아닙니다.


---

## 필수 1. 필드 설계와 HTML 데이터 추출

### 문제 1-1. 명언 하나를 하나의 레코드로 만들기

#### 문제 설명

명언의 본문을 살펴보고 작성자·태그별로 구분할 수 있는 데이터셋을 준비하려고 합니다. 교안의 가상 HTML인 `html_text`에서 필요한 값만 가져와 명언별 Dictionary로 정리하세요. 아직 단어 분석이나 작성자별 집계를 수행하는 문제는 아닙니다.

#### 요구사항

1. 최종 필드를 `text`, `author`, `tags`, `source_url`로 정하고, 각 필드의 의미·값을 가져올 위치·최종 저장 값의 형태를 답변란의 표에 작성하세요. 한 행의 의미도 적으세요.
2. `BeautifulSoup(html_text, "html.parser")`로 `soup`을 만들고, `select("div.quote")`로 모든 명언 영역을 찾으세요.
3. 각 영역 안에서 `select_one()`과 `get_text(strip=True)`로 본문·작성자를 추출하세요. `select("a.tag")`로 태그를 모두 찾아 텍스트 목록으로 만드세요.
4. 태그 목록은 `", ".join(tags)`로 한 문자열로 바꾸세요. `source_url`에는 `HTML_SOURCE_URL`을 사용하세요.
5. 네 Key를 가진 Dictionary를 명언마다 하나씩 만들고, 빈 List에서 시작한 `html_records`에 `append()`로 넣으세요.
6. 추출 건수와 명언별 Dictionary를 모두 출력하세요. Q1~Q4에 답하세요.

#### 해석 질문

**Q1.** 이번 데이터셋의 한 행은 무엇인가요? 첫 번째 명언에 태그가 두 개 있으면 행도 두 개가 되어야 하나요?

**Q2.** `select()`와 `select_one()`은 어떻게 다른가요? 명언 영역과 태그에 `select()`를 사용하는 이유는 무엇인가요?

**Q3.** 각 명언 영역 안에서 본문과 작성자를 추출하는 이유는 무엇인가요? `get_text(strip=True)`는 어떤 값을 반환하나요?

**Q4.** `tags`를 문자열로 바꾸는 이유와 `source_url`을 추가하는 이유를 각각 설명하세요.

#### 제출 결과

- HTML 추출 코드와 추출 건수·레코드 출력
- 필드 설계표와 한 행의 의미
- Q1~Q4 답변


### 작성 도움말 — 필수 1

한 번에 전체 코드를 쓰기 어렵다면 **명언 영역 찾기 → 영역 하나의 본문·작성자 찾기 → 태그 목록 만들기 → 레코드 추가**로 나누어 생각하세요. select_one()의 결과인 HTML 요소와 get_text()의 결과인 문자열을 구분하는 것이 핵심입니다.

- 선택자는 준비 부분의 네 가지를 사용합니다. 현재 명언 안의 정보는 quote를 기준으로 찾습니다.
- 결과 목록을 만드는 줄은 명언 반복문 앞에, 레코드를 추가하는 줄은 반복문 안에 둡니다.
- 필드 설계표에는 이름만 반복하지 말고, 어떤 값이며 어디에서 꺼내 어떤 형태로 담는지 적습니다.
- 실행 후에는 레코드 수와 원본 명언 영역 수가 대응하는지, 본문과 작성자가 같은 명언에 속하는지 확인하세요.


### 코드 작성란

아래 셀에 코드를 작성하고 실행 결과가 보이도록 제출하세요.


In [2]:
# 필수 1. 필드 설계와 HTML 데이터 추출 코드를 작성하세요.
# 위 요구사항의 변수명과 파일명을 사용하세요.

soup = BeautifulSoup(html_text, "html.parser")
quotes = soup.select("div.quote")

html_records = []

for quote in quotes:
    text = quote.select_one("span.text").get_text(strip=True)
    author = quote.select_one("small.author").get_text(strip=True)
    tags = [tag.get_text(strip=True) for tag in quote.select("a.tag")]

    record = {
        "text": text,
        "author": author,
        "tags": ", ".join(tags),
        "source_url": HTML_SOURCE_URL,
    }
    html_records.append(record)

print("추출 건수:", len(html_records))
for record in html_records:
    print(record)

추출 건수: 2
{'text': '첫 번째 예시 명언입니다.', 'author': '작성자 A', 'tags': 'life, thinking', 'source_url': 'https://quotes.toscrape.com/'}
{'text': '두 번째 예시 명언입니다.', 'author': '작성자 B', 'tags': 'learning', 'source_url': 'https://quotes.toscrape.com/'}


### 답변 작성란


- 한 행의 의미: 명언 하나

| 최종 필드 | 의미·사용 목적 | 값을 가져올 위치 | 최종 저장 값의 형태 |
|---|---|---|---|
| `text` | 명언 본문, 본문 분석에 사용 | div.quote 안의 span.text | 문자열 |
| `author` | 작성자 이름, 작성자별 구분에 사용 | div.quote 안의 small.author | 문자열 |
| `tags` | 관련 태그, 태그별 구분에 사용 | div.quote 안의 a.tag 전부 | 쉼표와 공백으로 연결한 하나의 문자열 |
| `source_url` | 데이터 출처 기록(메타데이터) | 원본에 없음, 프로그램에서 HTML_SOURCE_URL로 추가 | 문자열 |


**Q1. 이번 데이터셋의 한 행은 무엇인가요? 첫 번째 명언에 태그가 두 개 있으면 행도 두 개가 되어야 하나요?**

- 답변: 한 행은 명언 하나입니다. 첫 번째 명언에 태그가 두 개 있어도 "life, thinking"처럼 한 문자열로 묶어 tags 한 칸에 넣기 때문에 행은 하나입니다. 태그 수만큼 행을 늘리면 같은 명언이 두 번 세어질 것 같습니다.

**Q2. `select()`와 `select_one()`은 어떻게 다른가요? 명언 영역과 태그에 `select()`를 사용하는 이유는 무엇인가요?**

- 답변: `select()`는 조건에 맞는 요소를 모두 찾아 list로 돌려주고, `select_one()`은 첫 번째 요소 하나만 돌려줍니다. 명언 영역은 여러 개이고 한 명언 안의 태그도 여러 개일 수 있어서 모두 가져오려고 `select()`를 사용합니다.

**Q3. 각 명언 영역 안에서 본문과 작성자를 추출하는 이유는 무엇인가요? `get_text(strip=True)`는 어떤 값을 반환하나요?**

- 답변: 현재 명언 영역 안에서 찾아야 본문과 작성자가 같은 명언끼리 연결되기 때문입니다. `get_text(strip=True)`는 요소 안의 글자만 양쪽 공백을 제거한 문자열로 돌려줍니다.

**Q4. `tags`를 문자열로 바꾸는 이유와 `source_url`을 추가하는 이유를 각각 설명하세요.**

- 답변: 태그 목록을 문자열로 바꾸면 명언 하나가 한 행에 들어가고 CSV 한 칸에도 저장하기 쉬워집니다. `source_url`은 원본 HTML에는 없지만 나중에 이 데이터를 어디에서 가져왔는지 확인하려고 추가합니다.


---

## 필수 2. JSON 추출과 DataFrame·CSV 저장

### 문제 2-1. 같은 데이터를 공통 구조로 정리하고 저장 결과 확인하기

#### 문제 설명

`json_text`는 필수 1과 같은 가상 명언 두 개를 JSON으로 표현한 것입니다. 필요한 값만 추출하여 HTML 결과와 비교하고, JSON에서 얻은 레코드만 DataFrame으로 만들어 저장하세요.

#### 요구사항

1. `json.loads(json_text)`의 결과를 `data`에 저장하세요. `data["quotes"]`를 반복하여 `json_records`를 만드세요.
2. 본문은 `quote["text"]`, 작성자 이름은 `quote["author"]["name"]`에서 추출하세요. 태그는 쉼표와 공백으로 연결한 문자열로 만들고, 출처에는 `JSON_SOURCE_URL`을 사용하세요.
3. 각 Dictionary의 Key를 `text`, `author`, `tags`, `source_url`로 통일하고 추출 건수와 모든 레코드를 출력하세요.
4. `fields = ["text", "author", "tags"]`를 사용해 HTML·JSON 레코드에서 이 세 필드만 담은 `html_values`, `json_values`를 만드세요. 두 목록의 `==` 비교 결과를 출력하세요. 교안처럼 List·Dictionary 구성으로 비교합니다.
5. `columns = ["text", "author", "tags", "source_url"]`로 컬럼 순서를 정하세요. `json_records`만 `pd.DataFrame()`에 넣어 `df`를 만들고 `head()`, `shape`, `columns.tolist()`, `info()`로 기본 구조를 확인하세요.
6. `file_path = Path("quotes_example.csv")`를 사용하여 `df`를 저장하세요. 옵션은 `index=False`, `encoding="utf-8-sig"`로 지정하세요.
7. 같은 파일을 `pd.read_csv()`와 같은 인코딩으로 읽어 `loaded_df`에 저장하세요. `head()`, `shape`, `columns.tolist()`, `info()`를 확인하고, 저장 전후 행 수·컬럼 목록 일치 여부·`df.equals(loaded_df)` 결과를 출력하세요.
8. 답변란의 저장 결과 확인표를 채우고 Q1~Q5에 답하세요.

#### 해석 질문

**Q1.** `quote["author"]` 전체가 아니라 `quote["author"]["name"]`을 저장하는 이유는 무엇인가요?

**Q2.** HTML·JSON 비교에서 `source_url`을 제외하는 이유는 무엇인가요? 두 결과를 합쳐 저장하면 어떤 문제가 생기나요?

**Q3.** List 안의 Dictionary는 DataFrame의 행·열과 어떻게 연결되나요? `(2, 4)`와 `columns` 지정은 각각 무엇을 의미하나요?

**Q4.** CSV 저장 시 `index=False`와 `encoding="utf-8-sig"`를 사용하는 이유는 무엇인가요?

**Q5.** 행 수만 같으면 저장이 올바르다고 판단할 수 있나요? `equals()`가 True인 이번 결과와 CSV 자료형 보존의 한계를 설명하세요.

#### 제출 결과

- JSON 추출 코드와 결과, HTML·JSON 주요 필드 비교 결과
- DataFrame 생성·저장·다시 읽기 코드와 출력
- 실행하여 생성한 quotes_example.csv
- 저장 결과 확인표와 Q1~Q5 답변


### 작성 도움말 — 필수 2

이번 문제는 **JSON 추출 → HTML과 내용 비교 → DataFrame 생성 → CSV 저장 → 다시 읽기 → 비교**의 순서입니다. 각 단계의 결과를 확인한 뒤 다음 단계로 진행하세요.

- json.loads() 뒤에 quotes 목록을 꺼내야 명언을 하나씩 반복할 수 있습니다.
- 작성자는 정보 묶음 안의 name까지 접근합니다. 태그는 목록을 한 문자열로 연결합니다.
- 비교용 fields와 저장용 columns의 역할을 구분하세요. 비교에서 제외한 출처도 최종 표에는 필요합니다.
- 레코드·dict·list가 낯설다면, 먼저 한 명언의 네 값을 종이에 적고 그 묶음들이 목록에 들어간다고 생각하세요.
- 결과 확인표의 ‘결과의 의미’에는 무엇과 무엇을 비교했는지 적으세요. True라고만 적으면 HTML·JSON 비교인지 저장 전후 비교인지 알기 어렵습니다.


### 코드 작성란

아래 셀에 코드를 작성하고 실행 결과가 보이도록 제출하세요.


In [3]:
# 필수 2. JSON 추출과 DataFrame·CSV 저장 코드를 작성하세요.
# 위 요구사항의 변수명과 파일명을 사용하세요.

data = json.loads(json_text)

json_records = []
for quote in data["quotes"]:
    record = {
        "text": quote["text"],
        "author": quote["author"]["name"],
        "tags": ", ".join(quote["tags"]),
        "source_url": JSON_SOURCE_URL,
    }
    json_records.append(record)

print("JSON 추출 건수:", len(json_records))
for record in json_records:
    print(record)

# HTML·JSON 주요 필드 비교
fields = ["text", "author", "tags"]
html_values = [{field: record[field] for field in fields} for record in html_records]
json_values = [{field: record[field] for field in fields} for record in json_records]
print("HTML·JSON 주요 필드 일치:", html_values == json_values)

# DataFrame 생성
columns = ["text", "author", "tags", "source_url"]
df = pd.DataFrame(json_records, columns=columns)
print(df.head())
print("Shape:", df.shape)
print("컬럼:", df.columns.tolist())
df.info()

# CSV 저장
file_path = Path("quotes_example.csv")
df.to_csv(file_path, index=False, encoding="utf-8-sig")

# 다시 읽어 확인
loaded_df = pd.read_csv(file_path, encoding="utf-8-sig")
print(loaded_df.head())
print("다시 읽은 Shape:", loaded_df.shape)
print("다시 읽은 컬럼:", loaded_df.columns.tolist())
loaded_df.info()

print("저장 전 / 다시 읽은 행 수:", len(df), "/", len(loaded_df))
print("컬럼 목록 일치:", df.columns.tolist() == loaded_df.columns.tolist())
print("데이터 일치(df.equals):", df.equals(loaded_df))

JSON 추출 건수: 2
{'text': '첫 번째 예시 명언입니다.', 'author': '작성자 A', 'tags': 'life, thinking', 'source_url': 'https://quotes.toscrape.com/api/quotes?page=1'}
{'text': '두 번째 예시 명언입니다.', 'author': '작성자 B', 'tags': 'learning', 'source_url': 'https://quotes.toscrape.com/api/quotes?page=1'}
HTML·JSON 주요 필드 일치: True
             text author            tags  \
0  첫 번째 예시 명언입니다.  작성자 A  life, thinking   
1  두 번째 예시 명언입니다.  작성자 B        learning   

                                      source_url  
0  https://quotes.toscrape.com/api/quotes?page=1  
1  https://quotes.toscrape.com/api/quotes?page=1  
Shape: (2, 4)
컬럼: ['text', 'author', 'tags', 'source_url']
<class 'pandas.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   text        2 non-null      str  
 1   author      2 non-null      str  
 2   tags        2 non-null      str  
 3   source_url  2 non-null      str  
dtypes: str(4)
memory usage: 

### 답변 작성란


| 확인 항목 | 실행 결과 | 결과의 의미 |
|---|---|---|
| JSON 추출 건수 | 2 | JSON의 quotes 목록 안 명언 2개를 모두 추출함 |
| 주요 필드 일치 여부 | True | HTML과 JSON에서 꺼낸 text·author·tags 세 필드 값이 같음 |
| 저장 전 / 다시 읽은 Shape | (2, 4) / (2, 4) | 저장 전후 모두 2행 4열 |
| 저장 전 / 다시 읽은 행 수 | 2 / 2 | 저장 과정에서 행이 빠지거나 늘지 않음 |
| 컬럼 목록과 순서 | ['text', 'author', 'tags', 'source_url'] | 처음 정한 순서 그대로 |
| 컬럼 일치 여부 | True | 저장 전후 컬럼 이름과 순서가 같음 |
| 데이터 일치 여부 | True | 저장 전 df와 다시 읽은 loaded_df의 값이 같음 |
| `info()`로 확인한 값의 형태 | 네 컬럼 모두 2 non-null, object(문자열, pandas 3 이상에서는 str로 표시) | 네 컬럼 모두 결측 없이 문자열 형태 |


**Q1. `quote["author"]` 전체가 아니라 `quote["author"]["name"]`을 저장하는 이유는 무엇인가요?**

- 답변: `quote["author"]`는 `{"name": "작성자 A"}`라는 dict 전체입니다. 최종 author 컬럼에는 이름 문자열만 넣기로 정했기 때문에 한 단계 더 들어가서 `["name"]`까지 꺼내야 하고, 그래야 HTML 결과의 작성자와 형태도 같아진다고 생각합니다.

**Q2. HTML·JSON 비교에서 `source_url`을 제외하는 이유는 무엇인가요? 두 결과를 합쳐 저장하면 어떤 문제가 생기나요?**

- 답변: source_url은 HTML과 JSON에서 일부러 다른 주소를 기록했기 때문에, 포함하면 내용이 같은 명언이어도 False가 나옵니다. 그래서 내용만 비교하려고 제외했습니다. 두 결과를 합쳐 저장하면 같은 명언이 두 번씩 들어가 중복이 생길 것 같습니다.

**Q3. List 안의 Dictionary는 DataFrame의 행·열과 어떻게 연결되나요? `(2, 4)`와 `columns` 지정은 각각 무엇을 의미하나요?**

- 답변: List 안의 dict 하나가 DataFrame의 한 행이 되고, dict의 Key가 컬럼이 됩니다. `(2, 4)`는 2행 4열, 즉 명언 2개와 필드 4개라는 뜻이고, `columns` 지정은 컬럼의 순서를 고정하는 역할을 합니다.

**Q4. CSV 저장 시 `index=False`와 `encoding="utf-8-sig"`를 사용하는 이유는 무엇인가요?**

- 답변: `index=False`는 DataFrame 왼쪽의 0, 1 인덱스가 CSV에 불필요한 컬럼으로 저장되지 않게 합니다. `encoding="utf-8-sig"`는 한글이 깨지지 않게 저장하고 읽기 위한 설정입니다.

**Q5. 행 수만 같으면 저장이 올바르다고 판단할 수 있나요? `equals()`가 True인 이번 결과와 CSV 자료형 보존의 한계를 설명하세요.**

- 답변: 행 수만 같다고 올바르다고 판단할 수는 없습니다. 값이 바뀌거나 컬럼 순서가 달라졌을 수 있기 때문입니다. 이번에는 `equals()`가 True라서 값까지 같다는 걸 확인했습니다. 다만 CSV는 글자로만 저장되어 자료형 정보가 남지 않기 때문에, 숫자처럼 보이는 문자열이나 빈 칸은 다시 읽을 때 형태가 달라질 수 있다는 한계가 있습니다.


---

## 과제 1. 다른 HTML에 추출·저장 과정 적용

### 문제 3-1. 명언 세 개를 CSV로 저장하고 검증하기

#### 문제 설명

준비 셀의 `assignment_html`에는 교안 연습문제의 명언 A·B·C가 있습니다. 필수 1의 HTML 추출 방법과 필수 2의 저장 방법을 연결하여 독립적으로 처리하세요. 명언이 세 개라는 점을 제외하면 사용 방법은 같습니다.

#### 요구사항

1. `assignment_html`을 BeautifulSoup으로 해석하여 `assignment_soup`에 저장하세요.
2. 필수 1과 같은 선택자·추출 방법으로 모든 명언을 처리하여 `assignment_records`를 만드세요. Key는 `text`, `author`, `tags`, `source_url`로 통일하고, 태그는 쉼표와 공백으로 연결하세요. 출처에는 `HTML_SOURCE_URL`을 사용하세요.
3. 추출 건수와 모든 레코드를 출력하세요. `assignment_columns = ["text", "author", "tags", "source_url"]` 순서로 `assignment_df`를 만드세요.
4. `head()`, `shape`, `columns.tolist()`, `info()`로 DataFrame의 내용과 기본 구조를 확인하세요.
5. `assignment_path = Path("quotes_assignment.csv")`를 사용하여 `index=False`, `encoding="utf-8-sig"`로 저장하세요.
6. 같은 파일을 같은 인코딩으로 다시 읽어 `assignment_loaded_df`에 저장하세요. `head()`, `shape`, `columns.tolist()`, `info()`를 확인하고, 저장 전후 행 수·컬럼 일치 여부·데이터 일치 여부를 출력하세요.
7. 아래 결과 확인표와 Q1~Q4에 답하세요. 작성자가 같아도 서로 다른 명언은 각각 한 행으로 유지하세요.

#### 해석 질문

**Q1.** 과제 결과는 몇 행 몇 열인가요? 작성자 1이 두 번 나오는데도 두 명언을 각각 한 행으로 유지하는 이유는 무엇인가요?

**Q2.** 명언 A와 명언 C의 `tags`에 각각 어떤 문자열이 저장되나요? 태그 개수와 행 수를 어떻게 구분하나요?

**Q3.** 이 데이터에서 `text`가 비어 있다면 분석 목적상 무엇을 확인해야 하나요? 이번 과제에서 누락값 처리 코드까지 작성해야 하나요?

**Q4.** 예상한 3행과 달리 다시 읽은 파일이 2행이라면 어떤 순서로 확인하겠나요? 행 수 외에는 무엇을 확인해야 하나요?

#### 제출 결과

- 과제용 HTML 추출부터 CSV 재확인까지의 코드와 출력
- 실행하여 생성한 quotes_assignment.csv
- 결과 확인표와 Q1~Q4 답변


### 작성 도움말 — 과제 1

필수 1의 추출과 필수 2의 저장을 과제 입력에 적용합니다. 원본을 assignment_html로 바꾸는 것과 각 단계의 결과를 과제용 변수에 연결하는 것을 함께 확인하세요.

- 입력, 레코드 목록, DataFrame, 저장 경로, 다시 읽은 DataFrame을 순서대로 점검합니다.
- 저장 전의 표를 확인한 다음 파일을 쓰고, 같은 파일을 다시 읽습니다.
- 행 수를 셀 때 작성자 종류 수나 태그 개수와 혼동하지 마세요. 처음 정한 한 행의 기준으로 설명합니다.
- 예상과 다른 결과가 나오면 입력 → 추출 → 표 → 저장 → 읽기 중 처음 달라지는 지점을 찾습니다.


### 코드 작성란

아래 셀에 코드를 작성하고 실행 결과가 보이도록 제출하세요.


In [4]:
# 과제 1. 다른 HTML에 추출·저장 과정 적용 코드를 작성하세요.
# 위 요구사항의 변수명과 파일명을 사용하세요.

assignment_soup = BeautifulSoup(assignment_html, "html.parser")

assignment_records = []
for quote in assignment_soup.select("div.quote"):
    text = quote.select_one("span.text").get_text(strip=True)
    author = quote.select_one("small.author").get_text(strip=True)
    tags = [tag.get_text(strip=True) for tag in quote.select("a.tag")]

    assignment_records.append({
        "text": text,
        "author": author,
        "tags": ", ".join(tags),
        "source_url": HTML_SOURCE_URL,
    })

print("추출 건수:", len(assignment_records))
for record in assignment_records:
    print(record)

assignment_columns = ["text", "author", "tags", "source_url"]
assignment_df = pd.DataFrame(assignment_records, columns=assignment_columns)
print(assignment_df.head())
print("Shape:", assignment_df.shape)
print("컬럼:", assignment_df.columns.tolist())
assignment_df.info()

assignment_path = Path("quotes_assignment.csv")
assignment_df.to_csv(assignment_path, index=False, encoding="utf-8-sig")

assignment_loaded_df = pd.read_csv(assignment_path, encoding="utf-8-sig")
print(assignment_loaded_df.head())
print("다시 읽은 Shape:", assignment_loaded_df.shape)
print("다시 읽은 컬럼:", assignment_loaded_df.columns.tolist())
assignment_loaded_df.info()

print("저장 전 / 다시 읽은 행 수:", len(assignment_df), "/", len(assignment_loaded_df))
print("컬럼 일치:", assignment_df.columns.tolist() == assignment_loaded_df.columns.tolist())
print("데이터 일치:", assignment_df.equals(assignment_loaded_df))

추출 건수: 3
{'text': '명언 A', 'author': '작성자 1', 'tags': 'life, learning', 'source_url': 'https://quotes.toscrape.com/'}
{'text': '명언 B', 'author': '작성자 2', 'tags': 'thinking', 'source_url': 'https://quotes.toscrape.com/'}
{'text': '명언 C', 'author': '작성자 1', 'tags': 'success, life', 'source_url': 'https://quotes.toscrape.com/'}
   text author            tags                    source_url
0  명언 A  작성자 1  life, learning  https://quotes.toscrape.com/
1  명언 B  작성자 2        thinking  https://quotes.toscrape.com/
2  명언 C  작성자 1   success, life  https://quotes.toscrape.com/
Shape: (3, 4)
컬럼: ['text', 'author', 'tags', 'source_url']
<class 'pandas.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   text        3 non-null      str  
 1   author      3 non-null      str  
 2   tags        3 non-null      str  
 3   source_url  3 non-null      str  
dtypes: str(4)
memory usage: 228.0 bytes
   text

### 답변 작성란


| 확인 항목 | 실행 결과 |
|---|---|
| 추출 건수 | 3 |
| 저장 전 / 다시 읽은 Shape | (3, 4) / (3, 4) |
| 저장 전 / 다시 읽은 행 수 | 3 / 3 |
| 컬럼 목록과 순서 | ['text', 'author', 'tags', 'source_url'] |
| 컬럼 일치 여부 | True |
| 데이터 일치 여부 | True |
| `info()`로 확인한 값의 형태 | 네 컬럼 모두 3 non-null, object(문자열, pandas 3 이상에서는 str로 표시) |


**Q1. 과제 결과는 몇 행 몇 열인가요? 작성자 1이 두 번 나오는데도 두 명언을 각각 한 행으로 유지하는 이유는 무엇인가요?**

- 답변: 3행 4열입니다. 이번 데이터의 한 행 기준은 작성자가 아니라 명언 하나라서, 작성자 1이 두 번 나와도 명언 A와 명언 C는 본문이 다른 별개의 명언이므로 각각 한 행으로 유지해야 합니다.

**Q2. 명언 A와 명언 C의 `tags`에 각각 어떤 문자열이 저장되나요? 태그 개수와 행 수를 어떻게 구분하나요?**

- 답변: 명언 A는 "life, learning", 명언 C는 "success, life"가 저장됩니다. 태그 개수는 tags 한 칸 안에 쉼표로 이어진 태그 수이고, 행 수는 명언의 개수입니다. 그래서 태그가 2개여도 행은 1개입니다.

**Q3. 이 데이터에서 `text`가 비어 있다면 분석 목적상 무엇을 확인해야 하나요? 이번 과제에서 누락값 처리 코드까지 작성해야 하나요?**

- 답변: 이 데이터는 명언 본문이 핵심이라서, text가 비어 있으면 그 행을 본문 분석에 쓸 수 있는지, 원본 HTML에서 추출이 잘못된 건 아닌지 확인해야 한다고 생각합니다. 이번 과제는 추출·저장 단계라서 누락값 처리 코드까지는 작성하지 않아도 됩니다.

**Q4. 예상한 3행과 달리 다시 읽은 파일이 2행이라면 어떤 순서로 확인하겠나요? 행 수 외에는 무엇을 확인해야 하나요?**

- 답변: 입력 HTML의 div.quote 개수 → assignment_records 개수 → assignment_df 행 수 → 저장된 CSV → 다시 읽은 결과 순서로 처음 숫자가 달라지는 지점을 찾겠습니다. 행 수 외에도 컬럼 이름과 순서, 실제 값이 같은지 equals()로 확인해야 합니다.


---

## 실습 마무리

**Q1. 이번 실습에서 해결하려던 데이터 구성 문제는 무엇이었나요?**

- 답변: HTML과 JSON처럼 표현 방식이 다른 원본에서 명언 하나의 정보를 같은 네 필드로 꺼내 한 행으로 모으고, 표로 만들어 저장한 뒤 제대로 저장됐는지 확인하는 것이었습니다.

**Q2. HTML과 JSON에서 필요한 값을 꺼내는 방법은 각각 무엇이었나요?**

- 답변: HTML은 BeautifulSoup으로 파싱한 뒤 `select()`, `select_one()`으로 요소를 찾고 `get_text()`로 글자를 꺼냈습니다. JSON은 `json.loads()`로 파싱한 뒤 Key를 따라가며 값을 꺼냈습니다.

**Q3. 원본 데이터의 위치와 최종 컬럼명은 꼭 같아야 하나요? 작성자 이름을 예로 설명하세요.**

- 답변: 꼭 같지 않아도 됩니다. JSON에서 작성자 이름은 `author` 안의 `name`에 중첩되어 있었지만, 최종 컬럼명은 author 하나로 정하고 이름 문자열만 넣었습니다.

**Q4. 저장이 예상대로 이루어졌다고 판단한 근거는 무엇인가요?**

- 답변: 다시 읽은 CSV의 행 수와 컬럼이 저장 전과 같았고, `equals()` 결과가 True였기 때문입니다.

**Q5. CSV로 저장하는 것과 데이터 품질 검증·정제를 완료하는 것은 같은 의미인가요?**

- 답변: 같은 의미가 아니라고 생각합니다. 저장은 값을 파일에 기록한 것일 뿐이고, 결측·중복·이상한 값 같은 품질 검증과 정제는 따로 해야 합니다.

**Q6. 이번 예시 파일과 이후 교안의 quotes_raw.csv, quotes_clean.csv는 어떻게 다른가요?**

- 답변: 이번 파일은 교안의 가상 예시 명언으로 만든 연습용 파일입니다. 이후의 quotes_raw.csv는 실제 사이트에서 수집한 원시 데이터이고, quotes_clean.csv는 그 원시 데이터를 품질 확인과 정제까지 거친 결과입니다.


### 제출 전 확인

- 필수 1·2와 과제 1의 코드 및 출력 결과를 포함했는지 확인하세요.
- 필드 설계표·결과 확인표·해석 질문·실습 마무리에 모두 답했는지 확인하세요.
- `quotes_example.csv`, `quotes_assignment.csv`가 문제별로 만들어졌는지 확인하세요.
- 파일이 존재한다는 사실뿐 아니라 다시 읽은 행 수·컬럼·값을 확인했는지 점검하세요.
- 작성한 `.ipynb`와 실습에서 생성한 CSV 두 개를 제출하세요. CSV는 미리 제공되는 입력 파일이 아니라 직접 만드는 결과물입니다.


## 개념과 결과를 함께 정리하기

| 구분해야 할 것 | 이번 실습의 기준 |
|---|---|
| 구조 파악 / 값 추출 | 파싱으로 구조를 다루고, 선택자·Key로 필요한 값을 꺼냄 |
| 요소 / 문자열 | HTML 요소를 찾은 뒤 그 안의 글자를 get_text()로 얻음 |
| 레코드 / 레코드 목록 | 명언 하나는 dict, 여러 명언은 list |
| 원본 태그 / 저장할 태그 | 태그 목록을 join() 결과인 문자열로 한 칸에 정리 |
| 비교 필드 / 저장 컬럼 | 내용 비교는 세 필드, 저장은 출처를 포함한 네 컬럼 |
| 표 생성 / 파일 저장 | DataFrame은 메모리의 표, to_csv()가 파일 기록 |
| 파일 경로 / 파일 내용 | Path는 위치를 지정하고 read_csv()가 내용을 읽음 |
| 저장 전후 일치 / 품질 정제 | 같은 값이 저장되었는지 확인해도 모든 품질 문제가 해결되지는 않음 |

### 막혔을 때 먼저 확인할 부분

| 보이는 현상 | 먼저 확인할 것 |
|---|---|
| html_records라는 이름을 찾지 못함 | 준비와 필수 1을 먼저 실행했는지 |
| HTML에서 필요한 요소를 찾지 못함 | 현재 명언 영역과 선택자의 태그·클래스 이름이 맞는지 |
| JSON 항목 접근에서 오류가 남 | 현재 값이 전체 dict인지, 명언 list인지, 명언 하나의 dict인지 |
| 결과가 마지막 명언만 남음 | 결과 목록을 반복문 안에서 계속 새로 만들고 있지 않은지 |
| CSV가 보이지 않음 | 경로만 지정한 것이 아니라 to_csv()까지 실행했는지, 현재 작업 폴더가 어디인지 |
| 행 수는 같은데 비교가 False | 컬럼 이름·순서, 실제 값, 자료형을 함께 확인 |

학습 후에는 **“무엇을 한 행으로 정했는가, 어떤 위치에서 값을 꺼냈는가, 어떤 구조로 모았는가, 무엇으로 저장 결과를 확인했는가”**를 자신의 말로 설명해 보세요.
